# Phase 3 — Data Quality Validation with Great Expectations

Validates `silver.orders` directly against the in-cluster Spark DataFrame
(GX's Fluent Spark datasource), so no external connection or credentials
are needed. Four required expectations — not-null `order_id`, not-null
`customer_id`, `order_amount` strictly positive, `order_date` within the
dataset's known range — plus a stretch referential-integrity check that
every `customer_id` exists in `bronze.customers` (GX has no built-in FK
expectation, so this is done via `expect_column_values_to_be_in_set` fed
the distinct customer_id list).

In [ ]:
%pip install -q great_expectations

In [ ]:
dbutils.library.restartPython()

In [ ]:
import datetime

import great_expectations as gx
from great_expectations.checkpoint import UpdateDataDocsAction

## Set up a file-based Data Context

A file-based (rather than ephemeral) context is used so the generated
Data Docs are written to disk as a static HTML site.

In [ ]:
GX_PROJECT_ROOT = "/tmp/gx_project"

context = gx.get_context(mode="file", project_root_dir=GX_PROJECT_ROOT)

context.add_data_docs_site(
    site_name="local_site",
    site_config={
        "class_name": "SiteBuilder",
        "site_index_builder": {"class_name": "DefaultSiteIndexBuilder"},
        "store_backend": {
            "class_name": "TupleFilesystemStoreBackend",
            "base_directory": "uncommitted/data_docs/local_site/",
        },
    },
)

## Connect to silver.orders as a Spark dataframe batch

In [ ]:
data_source = context.data_sources.add_spark(name="silver_orders_datasource")
data_asset = data_source.add_dataframe_asset(name="silver_orders")
batch_definition = data_asset.add_batch_definition_whole_dataframe("silver_orders_batch")

silver_orders_df = spark.table("silver.orders")
batch_parameters = {"dataframe": silver_orders_df}

## Define the Expectation Suite

The 4 required expectations.

In [ ]:
suite = context.suites.add(gx.ExpectationSuite(name="silver_orders_suite"))

suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"))
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_amount", min_value=0, strict_min=True
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_date",
        min_value=datetime.date(2016, 1, 1),
        max_value=datetime.date(2018, 12, 31),
    )
)

## Stretch expectation — referential integrity

Every `customer_id` in Silver orders must exist in `bronze.customers`.
GX has no built-in foreign-key expectation, so this is approximated by
checking set membership against the distinct customer_id list — fine at
this dataset's scale, but collecting the full distinct list to the
driver wouldn't hold up on a much larger customers table.

In [ ]:
known_customer_ids = [
    row["customer_id"]
    for row in spark.table("bronze.customers").select("customer_id").distinct().collect()
]

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="customer_id", value_set=known_customer_ids
    )
)

## Validate — build a Validation Definition and run a Checkpoint

The Checkpoint's `UpdateDataDocsAction` regenerates the HTML Data Docs
with the new Validation Results after it runs.

In [ ]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="silver_orders_validation",
        data=batch_definition,
        suite=suite,
    )
)

checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="silver_orders_checkpoint",
        validation_definitions=[validation_definition],
        actions=[UpdateDataDocsAction(name="update_all_data_docs")],
    )
)

checkpoint_result = checkpoint.run(batch_parameters=batch_parameters)

## Pass/fail summary

In [ ]:
print(f"Checkpoint success: {checkpoint_result.success}\n")
print(checkpoint_result.describe())

## Data Docs

In [ ]:
data_docs_path = f"{GX_PROJECT_ROOT}/uncommitted/data_docs/local_site/index.html"

with open(data_docs_path, "r") as f:
    data_docs_html = f.read()

displayHTML(data_docs_html)